# AIFM Real Estate — Risk Notebook

Risk limits are defined in the fund's offering document and monitored against
internal thresholds. No regulatory VaR limit applies (unlike UCITS).

Key regulatory obligations under AIFMD:
- **Leverage**: gross and commitment method (Annex IV)
- **Stress testing**: market, liquidity, and counterparty scenarios (Annex VI)
- **Liquidity risk**: portfolio liquidity profile and redemption stress
- **Annex IV reporting**: quarterly to CSSF. AIFMD II (Directive 2024/927/EU)
  expanded requirements, adding granular data on liquidity management tools,
  loan origination, and delegation arrangements.

Regulatory framework:
- AIFMD: Directive 2011/61/EU
- AIFMD II: Directive 2024/927/EU
- Delegated Regulation: EU 231/2013
- Annex IV reporting: EU 231/2013, ESMA technical guidance v1.7 (July 2024)
- Annex VI stress testing: ESMA/2020/1498
- Luxembourg implementation: Law of 12 July 2013 on AIFMs (AIFM Law)

Dual UCITS/AIFM ManCo:
- CSSF Regulation 10-04 (organisational and prudential requirements)
- CSSF Regulation 22-05 (sustainability requirements, amending 10-04)

#### In this notebook

**AIFM Private RealEstate. Strategy: direct property investments (office, logistics,
retail, residential) across core European markets, complemented by listed REITs
for liquidity management. EUR and FX-hedged exposures.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.plot_style import ACCENT, ACCENT2, ACCENT3
import warnings
warnings.filterwarnings('ignore')

from src.setup_db import run as setup_db
setup_db()

from src.database import get_engine, query_positions, query_nav_history
from src.enrichment import get_risk_ready_df
from src.mock_bloomberg import MockBloomberg as Bloomberg
from src.leverage_config import INSTRUMENT_SOURCE
from src.risk_utils import (
    exception_report, full_backtest_report,
    stress_equity, stress_rates, stress_credit,
    stress_fx, stress_combined, stress_historical,
    stress_property, stress_rental, stress_ltv,
    days_to_liquidate, liquidity_buckets, redemption_stress,
    liquidity_adjusted_var,
)
from src.esg_utils import build_esg_df, esg_portfolio_summary, ESG_FIELDS, ESG_THRESHOLD

---

## 1. Load and Validate Positions

Positions are queried from SQLite, loaded daily from the fund administrator
Excel export. The flow is:

```
Fund admin Excel → load_positions() → SQLite → query_positions() → notebook
```

The Real Estate fund holds two types of assets with fundamentally different
data sources and risk characteristics:

- **Direct properties**: quarterly appraisal valuation, no Bloomberg ticker,
  fund admin data only (LTV, rental yield, vacancy rate, property type)
- **Listed REITs**: daily market prices, Bloomberg enrichment (beta, ADV)

`get_risk_ready_df` routes each position to the correct enrichment source
automatically based on the `is_direct_property` flag.


In [ ]:
positions = query_positions(ENGINE, FUND_ID, TODAY)
risk_df   = get_risk_ready_df(ENGINE, FUND_ID, TODAY)
NAV       = risk_df['market_value_eur'].sum()

direct    = risk_df[risk_df['is_direct_property'] == True]
listed    = risk_df[risk_df['is_direct_property'] == False]

print(f"Fund              : {FUND_ID}")
print(f"Valuation date    : {TODAY}")
print(f"Total positions   : {len(positions)}")
print(f"Direct properties : {len(direct)}")
print(f"Listed REITs/FX   : {len(listed)}")
print(f"NAV (EUR)         : {NAV:,.0f}")
print(f"Direct prop (EUR) : {direct['market_value_eur'].sum():,.0f}")
print(f"Listed (EUR)      : {listed['market_value_eur'].sum():,.0f}")

In [ ]:
# Asset class breakdown
breakdown = risk_df.groupby('asset_class').agg(
    market_value_eur=('market_value_eur', 'sum'),
    n_positions=('isin', 'count'),
).sort_values('market_value_eur', ascending=False)

breakdown['weight_pct'] = breakdown['market_value_eur'] / NAV * 100

print(f"{'Asset Class':<20} {'MV (EUR)':>15} {'Weight':>8} {'# Pos':>6}")
print('-' * 52)
for ac, row in breakdown.iterrows():
    print(f"{ac:<20} {row['market_value_eur']:>15,.0f} {row['weight_pct']:>7.1f}% {row['n_positions']:>6}")
print('-' * 52)
print(f"{'NAV':<20} {NAV:>15,.0f} {'100.0%':>8}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
colors = [ACCENT2 if v < 0 else ACCENT for v in breakdown['weight_pct']]
bars = ax.barh(breakdown.index, breakdown['weight_pct'],
               color=colors, height=0.45, alpha=0.85)
ax.axvline(0, color='#6b7280', lw=0.8)
ax.set_xlabel('Weight (% NAV)', fontsize=9)
ax.set_title('Asset Class Breakdown', color=ACCENT, fontsize=11, pad=12)
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax.grid(True, axis='x', alpha=0.15, linestyle='--')
ax.tick_params(labelsize=9, length=0)
for bar, val in zip(bars, breakdown['weight_pct']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---

## 2. VaR and Expected Shortfall

Direct properties are valued quarterly by independent appraisers. The absence of
daily pricing makes historical simulation VaR inapplicable to this portion of the
portfolio. AIFMD does not mandate VaR for real estate AIFs; regulatory risk
monitoring focuses on leverage, liquidity mismatch, and stress testing.

VaR is computed as follows:
- **Listed REITs**: historical simulation on daily NAV returns
- **Direct properties**: parametric VaR using an assumed annual volatility
  calibrated to MSCI Real Estate index data, disclosed in the RMP as an approximation
- **Combined**: liquid and illiquid VaR aggregated assuming independence

---

## 3. Direct Property Analysis

Key metrics for direct property positions: LTV, rental yield, and vacancy rate.
Sourced from the fund administrator quarterly valuation report.

- **LTV**: debt as % of property value. Covenant typically at 60-65%.
- **Rental yield**: annual rent / property value. Gross, before vacancy.
- **Vacancy rate**: % of lettable space not generating income.
- **Effective yield**: rental yield × (1 − vacancy rate). Actual income yield.

In [ ]:
prop_df = direct[direct['asset_class'] == 'Real Estate'].copy()
prop_df['effective_yield'] = prop_df['rental_yield_pct'] * (1 - prop_df['vacancy_rate_pct'] / 100)

summary = prop_df[[
    'instrument_name', 'country', 'market_value_eur',
    'ltv_pct', 'rental_yield_pct', 'vacancy_rate_pct',
    'effective_yield', 'property_type', 'valuation_date'
]].copy()

summary['weight_pct'] = summary['market_value_eur'] / NAV * 100

print(f"{'Property':<35} {'MV (EUR)':>12} {'Wgt':>6} {'LTV':>6} {'Yield':>7} {'Vacancy':>8} {'Eff.Yield':>10}")
print('-' * 92)
for _, row in summary.iterrows():
    print(f"{row['instrument_name']:<35} {row['market_value_eur']:>12,.0f} "
          f"{row['weight_pct']:>5.1f}% {row['ltv_pct']:>5.1f}% "
          f"{row['rental_yield_pct']:>6.1f}% {row['vacancy_rate_pct']:>7.1f}% "
          f"{row['effective_yield']:>9.1f}%")
print('-' * 92)

wav_ltv    = (summary['ltv_pct'] * summary['market_value_eur']).sum() / summary['market_value_eur'].sum()
wav_yield  = (summary['rental_yield_pct'] * summary['market_value_eur']).sum() / summary['market_value_eur'].sum()
wav_vac    = (summary['vacancy_rate_pct'] * summary['market_value_eur']).sum() / summary['market_value_eur'].sum()
wav_eff    = (summary['effective_yield'] * summary['market_value_eur']).sum() / summary['market_value_eur'].sum()
print(f"{'Weighted average':<35} {summary['market_value_eur'].sum():>12,.0f} "
      f"{'':>6} {wav_ltv:>5.1f}% {wav_yield:>6.1f}% {wav_vac:>7.1f}% {wav_eff:>9.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Direct Property Key Metrics', color=ACCENT, fontsize=11)

names = [n.split(' ')[0] + ' ' + n.split(' ')[1] for n in summary['instrument_name']]

for ax, col, title, limit in zip(
    axes,
    ['ltv_pct', 'rental_yield_pct', 'vacancy_rate_pct'],
    ['LTV (%)', 'Rental Yield (%)', 'Vacancy Rate (%)'],
    [60, None, None]
):
    colors = [ACCENT2 if (limit and v > limit) else ACCENT
              for v in summary[col]]
    bars = ax.barh(names, summary[col], color=colors,
                   height=0.45, alpha=0.85)
    if limit:
        ax.axvline(limit, color=ACCENT2, lw=1.2,
                   linestyle='--', label=f'Limit {limit}%')
        ax.legend(fontsize=7)
    ax.set_title(title, fontsize=9, color=ACCENT)
    ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
    ax.grid(True, axis='x', alpha=0.15, linestyle='--')
    ax.tick_params(labelsize=8, length=0)
    for bar, val in zip(bars, summary[col]):
        ax.text(bar.get_width() + 0.3,
                bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=8)

plt.tight_layout()
plt.show()

---

## 4. Leverage (Annex IV)

AIFMD requires leverage to be reported using two methods:

- **Gross method**: sum of absolute exposures divided by NAV. No netting allowed.
  Derivatives converted to equivalent underlying exposure.
- **Commitment method**: hedging and netting arrangements are recognised.
  Offsetting positions in the same underlying reduce exposure.

Limits are set in the fund's offering document and reported quarterly to the CSSF
in Annex IV. AIFMD II (Directive 2024/927/EU) expanded reporting requirements,
including the breakdown by:
* asset class
* instrument type
* source: financial borrowing, synthetic leverage through derivatives, and repo/reverse repo.

The expanded disclosure makes it easier for regulators to identify funds building leverage through
derivatives rather than borrowing.

In [ ]:
# MRS-23: Leverage computation - Gross and Commitment method

# ----------------------------------------------------------------
# Gross method: sum of absolute exposures / NAV
# ----------------------------------------------------------------
risk_df['abs_exposure'] = risk_df['market_value_eur'].abs()

deriv_rows     = risk_df[risk_df['asset_class'] == 'Derivative'].copy()
deriv_notional = 0.0
for _, row in deriv_rows.iterrows():
    ticker        = 'SPXW 260619P05500 Index'
    bbg_data      = BBG.bdp(ticker, ['DELTA', 'OPT_UNDL_PX', 'CONTRACT_SIZE'])
    delta         = abs(bbg_data.loc[ticker, 'DELTA'])
    undl_px       = bbg_data.loc[ticker, 'OPT_UNDL_PX']
    contract_size = bbg_data.loc[ticker, 'CONTRACT_SIZE']
    quantity      = abs(row['quantity'])
    fx_rate       = 0.89
    deriv_notional += delta * quantity * contract_size * undl_px * fx_rate

risk_df['gross_exposure'] = risk_df.apply(
    lambda r: deriv_notional if r['asset_class'] == 'Derivative'
    else (0.0 if r['asset_class'] == 'Cash'
    else r['abs_exposure']),
    axis=1
)

gross_leverage = risk_df['gross_exposure'].sum() / NAV

# ----------------------------------------------------------------
# Commitment method
# ----------------------------------------------------------------
long_eq  = risk_df[(risk_df['asset_class'] == 'Equity') &
                   (risk_df['market_value_eur'] > 0)]['market_value_eur'].sum()
short_eq = risk_df[(risk_df['asset_class'] == 'Equity') &
                   (risk_df['market_value_eur'] < 0)]['market_value_eur'].sum()
net_eq   = abs(long_eq + short_eq)
bonds    = risk_df[risk_df['asset_class'].isin(['Bond','Loan','CLO'])]['market_value_eur'].abs().sum()
fx       = risk_df[risk_df['asset_class'] == 'FX']['market_value_eur'].abs().sum()

commitment_exposure = net_eq + bonds + fx + deriv_notional
commitment_leverage = commitment_exposure / NAV

# ----------------------------------------------------------------
# Summary table
# ----------------------------------------------------------------
all_classes = sorted(risk_df['asset_class'].unique())

leverage_summary = pd.DataFrame({
    'Gross (EUR)'        : [risk_df[risk_df['asset_class']==ac]['gross_exposure'].sum()
                            for ac in all_classes],
    'Gross (x NAV)'      : [risk_df[risk_df['asset_class']==ac]['gross_exposure'].sum()/NAV
                            for ac in all_classes],
    'Commitment (EUR)'   : [risk_df[risk_df['asset_class']==ac]['market_value_eur'].abs().sum()
                            if ac not in ['Cash', 'Derivative'] else
                            (deriv_notional if ac == 'Derivative' else 0)
                            for ac in all_classes],
    'Commitment (x NAV)' : [risk_df[risk_df['asset_class']==ac]['market_value_eur'].abs().sum()/NAV
                            if ac not in ['Cash', 'Derivative'] else
                            (deriv_notional/NAV if ac == 'Derivative' else 0)
                            for ac in all_classes],
}, index=all_classes)

leverage_summary['Gross (EUR)']        = leverage_summary['Gross (EUR)'].map('{:,.0f}'.format)
leverage_summary['Gross (x NAV)']      = leverage_summary['Gross (x NAV)'].map('{:.2f}x'.format)
leverage_summary['Commitment (EUR)']   = leverage_summary['Commitment (EUR)'].map('{:,.0f}'.format)
leverage_summary['Commitment (x NAV)'] = leverage_summary['Commitment (x NAV)'].map('{:.2f}x'.format)

print(f"{'Asset Class':<15} {'Gross (EUR)':>15} {'Gross':>8} {'Commit (EUR)':>15} {'Commit':>8}")
print('-' * 65)
for ac in all_classes:
    row = leverage_summary.loc[ac]
    print(f"{ac:<15} {row['Gross (EUR)']:>15} {row['Gross (x NAV)']:>8} "
          f"{row['Commitment (EUR)']:>15} {row['Commitment (x NAV)']:>8}")
print('-' * 65)
print(f"{'Total':<15} {risk_df['gross_exposure'].sum():>15,.0f} {gross_leverage:>7.2f}x "
      f"{commitment_exposure:>15,.0f} {commitment_leverage:>7.2f}x")

GROSS_LIMIT = 3.0
status      = 'OK' if gross_leverage <= GROSS_LIMIT else 'BREACH'
print(f"\nGross leverage limit : {GROSS_LIMIT:.0f}x")
print(f"Current gross        : {gross_leverage:.2f}x")
print(f"Status               : {status}")

In [ ]:
# AIFMD II granular leverage breakdown
granular = risk_df.groupby(['asset_class', 'sub_asset_class']).agg(
    gross_eur=('gross_exposure', 'sum'),
    n_positions=('isin', 'count')
).reset_index()
granular['gross_x_nav'] = granular['gross_eur'] / NAV
granular['source']      = granular.apply(
    lambda r: INSTRUMENT_SOURCE.get(
        (r['asset_class'], r['sub_asset_class']), ('Other', 'Other'))[0], axis=1)
granular['listed_otc']  = granular.apply(
    lambda r: INSTRUMENT_SOURCE.get(
        (r['asset_class'], r['sub_asset_class']), ('Other', 'Other'))[1], axis=1)

granular = granular[granular['gross_eur'] > 0].sort_values('gross_eur', ascending=False)

# listed vs OTC summary
total_gross = granular['gross_eur'].sum()
summary_lot = granular.groupby('listed_otc')['gross_eur'].sum().reset_index()
summary_lot['x_nav']        = summary_lot['gross_eur'] / NAV
summary_lot['pct_leverage'] = summary_lot['gross_eur'] / total_gross * 100
summary_lot['gross_eur']    = summary_lot['gross_eur'].map('{:,.0f}'.format)
summary_lot['x_nav']        = summary_lot['x_nav'].map('{:.2f}x'.format)
summary_lot['pct_leverage'] = summary_lot['pct_leverage'].map('{:.1f}%'.format)
summary_lot.index.name      = None
summary_lot.columns         = ['Category', 'Gross (EUR)', 'x NAV', '% Leverage']
summary_lot.set_index('Category', inplace=True)

header = f"{'':12} {'Gross (EUR)':>15} {'x NAV':>8} {'% Leverage':>12}"
print(header)
print('-' * len(header))
for idx, row in summary_lot.iterrows():
    print(f"{idx:<12} {row['Gross (EUR)']:>15} {row['x NAV']:>8} {row['% Leverage']:>12}")
print('-' * len(header))
print()

summary_src = granular.groupby('source')['gross_eur'].sum().reset_index()
summary_src['x_nav']        = summary_src['gross_eur'] / NAV
summary_src['pct_leverage'] = summary_src['gross_eur'] / total_gross * 100
summary_src['gross_eur']    = summary_src['gross_eur'].map('{:,.0f}'.format)
summary_src['x_nav']        = summary_src['x_nav'].map('{:.2f}x'.format)
summary_src['pct_leverage'] = summary_src['pct_leverage'].map('{:.1f}%'.format)
summary_src.set_index('source', inplace=True)
summary_src.index.name      = None

header = f"{'':20} {'Gross (EUR)':>15} {'x NAV':>8} {'% Leverage':>12}"
print(header)
print('-' * len(header))
for idx, row in summary_src.iterrows():
    print(f"{idx:<20} {row['gross_eur']:>15} {row['x_nav']:>8} {row['pct_leverage']:>12}")
print('-' * len(header))
print()

# granular table
granular['pct_leverage'] = (granular['gross_eur'] / total_gross * 100).map('{:.1f}%'.format)
granular['gross_eur']    = granular['gross_eur'].map('{:,.0f}'.format)
granular['gross_x_nav']  = granular['gross_x_nav'].map('{:.2f}x'.format)
granular.set_index(['source', 'asset_class', 'sub_asset_class'], inplace=True)
granular

---

## 4. Stress Testing (Annex VI)

AIFMD Annex VI requires AIFMs to conduct regular stress tests covering market,
liquidity, and counterparty risk. Scenarios must be documented in the RMP,
reviewed at least annually, and results reported to the CSSF via Annex IV.

Unlike UCITS, no specific scenarios are prescribed. The scenarios below reflect
the specific risk drivers of a real estate fund: property valuation, rental
income, LTV covenants, and listed REIT market risk.

$$\Delta P_i = \text{sensitivity}_i \times \text{shock}_i \times MV_i$$

Scenarios covered:
- **Property value decline**: direct properties marked down 20%
- **Rental stress**: vacancy rate up 10pp, rental yield compressed 50bps
- **LTV covenant breach**: property values decline to LTV covenant threshold
- **Rate shock**: impact on listed REITs and floating rate debt
- **Historical**: 2008 financial crisis, 2011 EU sovereign debt crisis, 2020 COVID crash, 2022 rate shock

> **Methodology note**: in this project, stress P&L uses first-order sensitivities
> for listed REITs and direct revaluation for direct properties.
> In production these figures come from a third-party risk system or modeled to higher orders.


### Scenario assumptions
Here are the assumptions used in our liquidity tests.

| Scenario | Parameter | Value | 
|----------|-----------|-------|
| Property value stress | Office | -20% | 
| Property value stress | Logistics | -20% | 
| Property value stress | Retail | -25% | 
| Property value stress | Residential | -15% | 
| Rental stress | Vacancy rate increase | +10pp | 
| Rental stress | Rental yield compression | -50bps | 
| Rate shock | Parallel shift | +200bps | 
| LTV covenant breach | Threshold | 75% | 

Historical scenario shocks are defined in `HISTORICAL_SCENARIOS` in `src/risk_utils.py`, and are printed below.

In [ ]:
# MRS-26: Annex VI stress scenarios - real estate
from src.risk_utils import HISTORICAL_SCENARIOS, stress_property, stress_rental, stress_ltv

# scenario table
rows = []
for key, p in HISTORICAL_SCENARIOS.items():
    rows.append({
        'Scenario'    : p['name'],
        'Equity'      : f"{p['delta_equity']*100:.0f}%",
        'Rates (bps)' : f"{p['delta_y']*10000:.0f}",
        'Credit (bps)': f"+{p['delta_spread']*10000:.0f}",
        'USD'         : f"{p['fx_shocks'].get('USD', 0)*100:+.0f}%",
        'GBP'         : f"{p['fx_shocks'].get('GBP', 0)*100:+.0f}%",
    })
pd.DataFrame(rows).set_index('Scenario')

In [ ]:
prop = stress_property(risk_df, delta_value_by_type={
    'Office': -0.20, 'Logistics': -0.20, 'Retail': -0.25, 'Residential': -0.15
})
rent = stress_rental(risk_df, delta_vacancy=0.10, delta_yield=-0.005)
rt   = stress_rates(risk_df, delta_y=0.02)
hist = {s: stress_historical(risk_df, s) for s in HISTORICAL_SCENARIOS}

# LTV covenant breach - separate output
ltv = stress_ltv(risk_df, delta_property_value=-0.25) # assuming a severe property value shock of -25%
print(f"LTV covenant breaches: {ltv['n_breaches']}")
if ltv['n_breaches'] > 0:
    print(pd.DataFrame(ltv['breaching_properties']).to_string())
else:
    print("No LTV covenant breaches under current portfolio.")
print()

rows = [
    {'Scenario': 'Property Value Stress',   'P&L (EUR)': prop['stressed_pnl_eur'], '% NAV': prop['stressed_pnl_eur']/NAV*100},
    {'Scenario': 'Rental Stress',           'P&L (EUR)': rent['stressed_pnl_eur'], '% NAV': rent['stressed_pnl_eur']/NAV*100},
    {'Scenario': 'Rate Shock +200bps',      'P&L (EUR)': rt['stressed_pnl_eur'],   '% NAV': rt['stressed_pnl_eur']/NAV*100},
] + [
    {'Scenario': v['scenario'], 'P&L (EUR)': v['stressed_pnl_eur'], '% NAV': v['stressed_pnl_eur']/NAV*100}
    for v in hist.values()
]

summary_raw = pd.DataFrame(rows).set_index('Scenario')
worst_idx   = summary_raw['% NAV'].idxmin()

summary_raw['P&L (EUR)'] = summary_raw['P&L (EUR)'].map('{:,.0f}'.format)
summary_raw['% NAV']     = summary_raw['% NAV'].map('{:.2f}%'.format)

summary_raw.style.apply(lambda x: [
    'background-color: #7f1d1d; color: white' if i == worst_idx else ''
    for i in x.index], axis=0)

> **Note on rate shock**: `stress_rates` applies duration sensitivity to bonds
> and REITs only. For direct properties the relevant transmission mechanism is
> cap rate expansion (higher rates compress property valuations), which is not
> captured here. In practice a 200bps rate shock would also reduce direct
> property values by approximately 10-15% through cap rate widening.
> This is partially captured in the property value stress scenario above.

---
---
## 6. Liquidity Profile

ESMA guidelines (ESMA34-39-897) require AIFMs to categorise portfolio assets
by time to liquidation under normal market conditions. Results are reported
to the CSSF via Annex IV and used to assess redemption coverage.

Liquidity buckets (ESMA standard):

| Bucket | Time to liquidate |
|--------|------------------|
| 1      | 1 day            |
| 2      | 2-7 days         |
| 3      | 8-30 days        |
| 4      | 31-90 days       |
| 5      | 91-365 days      |
| 6      | > 1 year         |

ESMA buckets are roughly: day, week, month, quarter, year, or above.

Days to liquidate per position:

$$\text{days}_i = \frac{|MV_i|}{ADV_i \times \text{pct\_adv}}$$

* **ADV** (Average Daily Volume): average contracts/shares traded per day over
a 20-day window, sourced from Bloomberg. 
* **pct_adv = 25%** is an internal
RMP parameter representing the maximum fraction of ADV tradeable per day
without significant market impact. 
* Direct properties and instruments with
zero ADV are classified as illiquid (> 1 year).
* Cash and money market instruments are classified as 1 day by definition,
as they require no liquidation process.

---
## 6. Liquidity Profile

ESMA guidelines (ESMA34-39-897) require AIFMs to categorise portfolio assets
by time to liquidation under normal market conditions. Results are reported
to the CSSF via Annex IV and used to assess redemption coverage.

Liquidity buckets (ESMA standard):

| Bucket | Time to liquidate |
|--------|------------------|
| 1      | 1 day            |
| 2      | 2-7 days         |
| 3      | 8-30 days        |
| 4      | 31-90 days       |
| 5      | 91-365 days      |
| 6      | > 1 year         |

ESMA buckets are roughly: day, week, month, quarter, year, or above.

Days to liquidate per position:

$$\text{days}_i = \frac{|MV_i|}{ADV_i \times \text{pct\_adv}}$$

* **ADV** (Average Daily Volume): average contracts/shares traded per day over
a 20-day window, sourced from Bloomberg. 
* **pct_adv = 25%** is an internal
RMP parameter representing the maximum fraction of ADV tradeable per day
without significant market impact. 
* Direct properties and instruments with
zero ADV are classified as illiquid (> 1 year).
* Cash and money market instruments are classified as 1 day by definition,
as they require no liquidation process.

In [ ]:

# MRS-30: Liquidity profile
risk_df_liq = days_to_liquidate(risk_df, pct_adv=0.25)
risk_df_liq = liquidity_buckets(risk_df_liq)

bucket_order = ['1 day', '2-7 days', '8-30 days', '31-90 days', '91-365 days', '> 1 year']

bucket_summary = risk_df_liq.groupby('liquidity_bucket').agg(
    market_value_eur=('market_value_eur', 'sum'),
    abs_exposure=('market_value_eur', lambda x: x.abs().sum()),
    n_positions=('isin', 'count')
).reset_index()
bucket_summary['pct_nav_net'] = bucket_summary['market_value_eur'] / NAV * 100
bucket_summary['pct_nav_abs'] = bucket_summary['abs_exposure'] / NAV * 100

bucket_full = bucket_summary.set_index('liquidity_bucket').reindex(bucket_order).fillna(0).reset_index()

# table
print(f"{'Bucket':<15} {'Abs Exposure (EUR)':>20} {'% NAV':>8} {'# Pos':>6}")
print('-' * 55)
for _, row in bucket_full.iterrows():
    if row['abs_exposure'] > 0:
        print(f"{row['liquidity_bucket']:<15} {row['abs_exposure']:>20,.0f} "
              f"{row['pct_nav_abs']:>7.1f}% {row['n_positions']:>6.0f}")
    else:
        print(f"{row['liquidity_bucket']:<15} {'—':>20} {'—':>8} {'—':>6}")
print('-' * 55)
total_abs = risk_df_liq['market_value_eur'].abs().sum()
print(f"{'Total':<15} {total_abs:>20,.0f} {total_abs/NAV*100:>7.1f}%")


# chart
fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.bar(bucket_full['liquidity_bucket'],
              bucket_full['pct_nav_abs'],
              color=ACCENT, alpha=0.85, width=0.5)
ax.axhline(0, color='#6b7280', lw=0.8)
ax.set_ylabel('Absolute Exposure (% NAV)', fontsize=9)
ax.set_title('Liquidity Profile — Absolute Exposure by Bucket',
             color=ACCENT, fontsize=11, pad=12)
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax.grid(True, axis='y', alpha=0.15, linestyle='--')
ax.tick_params(labelsize=9, length=0)
for bar, val in zip(bars, bucket_full['pct_nav_abs']):
    if val > 2:
        ax.text(bar.get_x() + bar.get_width()/2, val/2,
                f'{val:.1f}%', ha='center', va='center',
                fontsize=10, color='white', fontweight='bold')
plt.tight_layout()
plt.show()



## ESG Risk Indicators

ESG monitoring is required under CSSF Regulation 22-05 and SFDR (EU 2019/2088).
Portfolio-level ESG scores are computed as NAV-weighted averages. 

**SFDR classification**: Article 6 (no sustainability claim). ESG factors are
monitored but do not drive investment decisions. Article 8 would require documented
ESG screening; Article 9 would require sustainable investment as the primary objective.

Metrics monitored:
- Weighted average ESG score (composite, E, S, G)
- % of NAV in instruments with ESG score below 30*
- % of NAV with active controversy flag
- Weighted average carbon intensity (tCO2/EURm revenue)

> **Note**: ESG scores for liquid instruments are fetched from Bloomberg at
> enrichment time. Instruments without a Bloomberg ticker use fund administrator
> ESG data embedded in the position file.


> % of NAV in instruments with ESG score below internal threshold. 
> Threshold is defined in the Risk Management Policy. 
> ESG scores are not comparable across providers as each
> uses a different methodology and scale.
> (here using 30/100,Bloomberg scale 0-100 higher is better)

> **Scale note**: all ESG scores in this framework use a 0-100 scale where
> 100 is best, consistent with Bloomberg ESG methodology. Illiquid instrument
> scores are provided by the fund administrator on the same scale. In production,
> the ManCo should ensure all third-party ESG data is normalised to a consistent
> scale before aggregating portfolio-level metrics.

> **ESG look-through for derivatives**: derivatives have indirect ESG exposure
> through their underlying. The delta-adjusted notional is used as the ESG
> exposure weight rather than market value, which would understate the exposure.
>
> For options:
> $$ESG\_exposure_i = |\delta_i| \times Q_i \times \text{contract size} \times P_{underlying} \times FX$$
>
> For futures: full notional is used since delta = 1.
>
> For FX forwards: no ESG exposure assigned (no issuer).

In [ ]:
esg_df  = build_esg_df(risk_df, BBG, ENGINE, FUND_ID, TODAY)
summary = esg_portfolio_summary(esg_df, NAV)

In [ ]:
def style_esg(row):
    styles = [''] * len(row)
    idx = esg_df.columns.tolist()
    if row.get('controversy_flag') == True:
        return ['background-color: #7f1d1d; color: white'] * len(row)
    elif pd.notna(row.get('esg_score')) and row.get('esg_score') < ESG_THRESHOLD:
        styles[idx.index('esg_score')] = 'background-color: #7f1d1d; color: white'
    return styles

esg_df.style.apply(style_esg, axis=1).format({
    'market_value_eur' : '{:,.0f}',
    'weight_pct'       : '{:.2f}%',
    'esg_score'        : '{:.0f}',
    'env_score'        : '{:.0f}',
    'soc_score'        : '{:.0f}',
    'gov_score'        : '{:.0f}',
    'carbon_intensity' : '{:.1f}',
    'esg_exposure_eur' : '{:,.0f}',
}, na_rep='—')

In [ ]:
print(f"ESG PORTFOLIO SUMMARY")
print('-' * 45)
print(f"{'Weighted avg ESG score':<30} {summary['wav_esg']:.1f}/100")
print(f"{'Weighted avg ENV score':<30} {summary['wav_env']:.1f}/100")
print(f"{'Weighted avg SOC score':<30} {summary['wav_soc']:.1f}/100")
print(f"{'Weighted avg GOV score':<30} {summary['wav_gov']:.1f}/100")
print(f"{'Weighted avg carbon intensity':<30} {summary['wav_carbon']:.1f} tCO2/EURm")
print(f"{'% exposure below ESG threshold':<30} {summary['pct_low_esg']:.1f}%")
print(f"{'% exposure with controversy':<30} {summary['pct_controversy']:.1f}%")
print()
if len(summary['controversies']) > 0:
    print("Controversy flags:")
    for _, row in summary['controversies'].iterrows():
        print(f"  {row['instrument_name']:<35} ESG: {row['esg_score']:.0f}")

In [ ]:
esg_scored = esg_df[esg_df['esg_score'].notna()].copy()
total_scored_mv = esg_scored['esg_exposure_eur'].sum()
ac_esg = esg_scored.groupby('asset_class').agg(
    wav_esg=('esg_score', lambda x: (x * esg_scored.loc[x.index, 'esg_exposure_eur']).sum() /
             esg_scored.loc[x.index, 'esg_exposure_eur'].sum()),
    exposure=('esg_exposure_eur', 'sum')
).reset_index()
ac_esg['pct_total'] = ac_esg['exposure'] / total_scored_mv * 100

fig, axes = plt.subplots(2, 1, figsize=(12, 5))
fig.suptitle('ESG Profile by Asset Class', color=ACCENT, fontsize=11)

colors = ['#9ca3af', '#6b7280', '#4b5563', '#374151', '#d1d5db', '#e5e7eb']
left = 0
for i, (_, row) in enumerate(ac_esg.iterrows()):
    axes[0].barh(0, row['pct_total'], left=left,
                 color=colors[i % len(colors)], alpha=0.85, height=0.4)
    if row['pct_total'] > 3:
        axes[0].text(left + row['pct_total']/2, 0,
                     f"{row['asset_class']}\n{row['pct_total']:.1f}%",
                     ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    left += row['pct_total']

axes[0].set_xlim(0, 100)
axes[0].set_yticks([])
axes[0].set_xlabel('% of ESG-scored exposure', fontsize=9)
axes[0].spines[['top', 'right', 'left', 'bottom']].set_visible(False)
axes[0].tick_params(labelsize=9, length=0)

bars = axes[1].barh(ac_esg['asset_class'], ac_esg['wav_esg'],
                    color=[ACCENT2 if v < 50 else ACCENT3 if v < 70 else ACCENT
                           for v in ac_esg['wav_esg']],
                    height=0.4, alpha=0.85)
axes[1].axvline(ESG_THRESHOLD, color=ACCENT2, lw=1, linestyle='--',
                label=f'Low ESG threshold ({ESG_THRESHOLD})')
axes[1].axvline(70, color=ACCENT3, lw=1, linestyle='--', label='Good ESG threshold (70)')
axes[1].set_xlim(0, 100)
axes[1].set_xlabel('Weighted avg ESG score', fontsize=9)
axes[1].spines[['top', 'right', 'left', 'bottom']].set_visible(False)
axes[1].grid(True, axis='x', alpha=0.15, linestyle='--')
axes[1].tick_params(labelsize=9, length=0)
axes[1].legend(fontsize=7)
for bar, val in zip(bars, ac_esg['wav_esg']):
    axes[1].text(val + 1, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()